In [1]:
# Check if the easydiffraction library is installed.
# If not, install it with the 'visualization' extras.
# Needed when running remotely (e.g. Colab) where the lib is absent.
import builtins
import importlib.util

if (hasattr(builtins, '__IPYTHON__') and
    importlib.util.find_spec('easydiffraction') is None):
    !pip install 'easydiffraction[visualization]'

# Joint Refinement: Si, Bragg + PDF

This example demonstrates a joint refinement of the Si crystal
structure combining Bragg diffraction and pair distribution function
(PDF) analysis. The Bragg experiment uses time-of-flight neutron
powder diffraction data from SEPD at Argonne, while the PDF
experiment uses data from NOMAD at SNS. A single shared Si structure
is refined simultaneously against both datasets.

## Import Library

In [2]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data

## Define Structure

A single Si structure is shared between the Bragg and PDF
experiments. Structural parameters refined against both datasets
simultaneously.

#### Create Structure

In [3]:
structure = StructureFactory.from_scratch(name='si')

#### Set Space Group

In [4]:
structure.space_group.name_h_m = 'F d -3 m'
structure.space_group.it_coordinate_system_code = '1'

#### Set Unit Cell

In [5]:
structure.cell.length_a = 5.42

#### Set Atom Sites

In [6]:
structure.atom_sites.create(
    label='Si',
    type_symbol='Si',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    b_iso=0.2,
)

## Define Experiments

Two experiments are defined: one for Bragg diffraction and one for
PDF analysis. Both are linked to the same Si structure.

### Experiment 1: Bragg (SEPD, TOF)

#### Download Data

In [7]:
bragg_data_path = download_data(id=7, destination='data')

Getting data...


Data #7: Si, SEPD (Argonne)


✅ Data #7 downloaded to 'data/ed-7.xye'


#### Create Experiment

In [8]:
bragg_expt = ExperimentFactory.from_data_path(
    name='sepd', data_path=bragg_data_path, beam_mode='time-of-flight'
)

Data loaded successfully


Experiment 🔬 'sepd'. Number of data points: 5600


#### Set Instrument

In [9]:
bragg_expt.instrument.setup_twotheta_bank = 144.845
bragg_expt.instrument.calib_d_to_tof_offset = -9.2
bragg_expt.instrument.calib_d_to_tof_linear = 7476.91
bragg_expt.instrument.calib_d_to_tof_quad = -1.54

#### Set Peak Profile

In [10]:
bragg_expt.peak_profile_type = 'pseudo-voigt * ikeda-carpenter'
bragg_expt.peak.broad_gauss_sigma_0 = 5.0
bragg_expt.peak.broad_gauss_sigma_1 = 45.0
bragg_expt.peak.broad_gauss_sigma_2 = 1.0
bragg_expt.peak.broad_mix_beta_0 = 0.04221
bragg_expt.peak.broad_mix_beta_1 = 0.00946
bragg_expt.peak.asym_alpha_0 = 0.0
bragg_expt.peak.asym_alpha_1 = 0.5971

⚠️ Switching peak profile type discards existing peak parameters.                                                                 


Peak profile type for experiment 'sepd' changed to


pseudo-voigt * ikeda-carpenter


#### Set Background

In [11]:
bragg_expt.background_type = 'line-segment'
for x in range(0, 35000, 5000):
    bragg_expt.background.create(id=str(x), x=x, y=200)

Background type for experiment 'sepd' already set to


line-segment


#### Set Linked Phases

In [12]:
bragg_expt.linked_phases.create(id='si', scale=13.0)

### Experiment 2: PDF (NOMAD, TOF)

#### Download Data

In [13]:
pdf_data_path = download_data(id=5, destination='data')

Getting data...


Data #5: NOM_9999_Si_640g_PAC_50_ff_ftfrgr_up-to-50.gr


✅ Data #5 already present at 'data/ed-5.gr'. Keeping existing file.


#### Create Experiment

In [14]:
pdf_expt = ExperimentFactory.from_data_path(
    name='nomad',
    data_path=pdf_data_path,
    beam_mode='time-of-flight',
    scattering_type='total',
)

Data loaded successfully


Experiment 🔬 'nomad'. Number of data points: 5033


#### Set Peak Profile (PDF Parameters)

In [15]:
pdf_expt.peak.damp_q = 0.02
pdf_expt.peak.broad_q = 0.02
pdf_expt.peak.cutoff_q = 35.0
pdf_expt.peak.sharp_delta_1 = 0.001
pdf_expt.peak.sharp_delta_2 = 4.0
pdf_expt.peak.damp_particle_diameter = 0

#### Set Linked Phases

In [16]:
pdf_expt.linked_phases.create(id='si', scale=1.0)

## Define Project

The project object manages the shared structure, both experiments,
and the analysis.

#### Create Project

In [17]:
project = Project()

#### Add Structure

In [18]:
project.structures.add(structure)

#### Add Experiments

In [19]:
project.experiments.add(bragg_expt)
project.experiments.add(pdf_expt)

## Perform Analysis

This section shows the joint analysis process. The calculator is
auto-resolved per experiment: CrysPy for Bragg, PDFfit for PDF.

#### Set Fit Mode and Weights

In [20]:
project.analysis.fit_mode.mode = 'joint'
project.analysis.joint_fit_experiments.create(id='sepd', weight=0.7)
project.analysis.joint_fit_experiments.create(id='nomad', weight=0.3)

#### Set Minimizer

In [21]:
project.analysis.current_minimizer = 'lmfit'

Current minimizer changed to


lmfit


#### Plot Measured vs Calculated (Before Fit)

In [22]:
project.plot_meas_vs_calc(expt_name='sepd', show_residual=False)

In [23]:
project.plot_meas_vs_calc(expt_name='nomad', show_residual=False)

#### Set Fitting Parameters

Shared structural parameters are refined against both datasets
simultaneously.

In [24]:
structure.cell.length_a.free = True
structure.atom_sites['Si'].b_iso.free = True

Bragg experiment parameters.

In [25]:
bragg_expt.linked_phases['si'].scale.free = True
bragg_expt.instrument.calib_d_to_tof_offset.free = True
bragg_expt.peak.broad_gauss_sigma_0.free = True
bragg_expt.peak.broad_gauss_sigma_1.free = True
bragg_expt.peak.broad_gauss_sigma_2.free = True
for point in bragg_expt.background:
    point.y.free = True

PDF experiment parameters.

In [26]:
pdf_expt.linked_phases['si'].scale.free = True
pdf_expt.peak.damp_q.free = True
pdf_expt.peak.broad_q.free = True
pdf_expt.peak.sharp_delta_1.free = True
pdf_expt.peak.sharp_delta_2.free = True

#### Show Free Parameters

In [27]:
project.analysis.show_free_params()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.42000,,-inf,inf,Å
2,si,atom_site,Si,b_iso,0.20000,,-inf,inf,Å²
3,sepd,linked_phases,si,scale,13.00000,,-inf,inf,
4,sepd,peak,,gauss_sigma_0,5.00000,,-inf,inf,µs²
5,sepd,peak,,gauss_sigma_1,45.00000,,-inf,inf,µs/Å
6,sepd,peak,,gauss_sigma_2,1.00000,,-inf,inf,µs²/Å²
7,sepd,instrument,,d_to_tof_offset,-9.20000,,-inf,inf,µs
8,sepd,background,0,y,200.00000,,-inf,inf,
9,sepd,background,5000,y,200.00000,,-inf,inf,
10,sepd,background,10000,y,200.00000,,-inf,inf,


#### Run Fitting

In [28]:
project.analysis.fit()
project.analysis.show_fit_results()

Using all experiments 🔬 ['sepd', 'nomad'] for 'joint' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit (reduced χ²) change:


,iteration,χ²,improvement [%]
1,1,3378.13,
2,23,844.35,75.0% ↓
3,43,267.05,68.4% ↓
4,63,60.34,77.4% ↓
5,83,52.57,12.9% ↓
6,103,51.89,1.3% ↓
7,167,51.87,


<IPython.core.display.Javascript object>

...

📏 Weighted R-factor (wR): 8.30%


📈 Fitted parameters:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,start,fitted,uncertainty,units,change
1,si,cell,,length_a,5.4200,5.4306,0.0000,Å,0.20 % ↑
2,si,atom_site,Si,b_iso,0.2000,0.7035,0.0037,Å²,251.76 % ↑
3,sepd,linked_phases,si,scale,13.0000,16.0536,0.1008,,23.49 % ↑
4,sepd,peak,,gauss_sigma_0,5.0000,-2.0906,1.2338,µs²,141.81 % ↓
5,sepd,peak,,gauss_sigma_1,45.0000,50.6833,2.4533,µs/Å,12.63 % ↑
6,sepd,peak,,gauss_sigma_2,1.0000,0.1783,0.4966,µs²/Å²,82.17 % ↓
7,sepd,instrument,,d_to_tof_offset,-9.2000,-8.2396,0.0950,µs,10.44 % ↓
8,sepd,background,0,y,200.0000,280.5930,3.2291,,40.30 % ↑
9,sepd,background,5000,y,200.0000,148.7112,1.3525,,25.64 % ↓
10,sepd,background,10000,y,200.0000,118.2945,1.4191,,40.85 % ↓


#### Plot Measured vs Calculated (After Fit)

In [29]:
project.plot_meas_vs_calc(expt_name='sepd', show_residual=False)

In [30]:
project.plot_meas_vs_calc(expt_name='nomad', show_residual=False)